« model_17 — DÜZ MATEMATİK (MAT) »

**Soru: DT (hedef mimari) toplamayı eski mimariden iyi mi öğreniyor — KURALI mı öğreniyor, ezberi mi?**
Önkayıt `belge/onkayit/model_17_MAT.md` — veri görülmeden yazıldı.

| ne | değer |
|---|---|
| veri | model_15 `veri_cok.pt` · 2+3 terim, her terim 0..500 · iz `8d0f89938c67e0e7` |
| biçim | `<eos> soru cevap <eos>` · kayıp YALNIZ cevabın rakamlarında ve EOS'ta |
| lr · yığın · adım | 0,002 · 4096 · 16.000 (kullanıcı, 23 Eylül) — 16.000 ilk sınır, tavan değil |
| ölçüm · yedek | ikisi de her **100** adımda → 161 nokta, her noktanın paketi var |
| her adımın kaybı | pakette `kayip_iz` · 16.001 sayı, bedelsiz |
| kollar | `MAT_COK_DT16` eş boy, 7.860 par · `MAT_COK_DT256` küçük, 1.207.044 par |
| tek düğmeli | `MAT_COK_DT16_B512` A: bellek 512, 37.428 par · `MAT_COK_DT16_Y2` B: yansıma 2, 8.918 par · `MAT_COK_DT16_GB16` geçici bellek W 16 × 4 kafa, 10.068 par — hepsi 1a'dan TEK fark |

**Taban** (eski mimari, 4.529 par, aynı veri, aynı ölçüt, 16.000 adım): tutulan **0,4403** · 2 terim 0,5121 · 3 terim **0,3693**.

**Karar kuralı** (önkayıt): DT 3 terimde > 0,3693 → "yol uzayabilir" eskisinden güçlü · eğitim − tutulan < 0,02 → ezber değil kural · ilk rakam ayrıca raporlanır, hüküm vermez. DUR kuralı (≥ 0,7770) bu koşudan **okunmaz**; o sayı yalnız 2 terimle eğitilmiş modelin.

**Neden her 100 adım, neden her adımın kaybı.** Tabanın eğrisi 400 adımda bir ölçülmüştü ve sert dalgalanıyor: 13.600'de 0,3543 · 14.000'de **0,0943** · 14.400'de 0,3161. Bu seyreklikte tek adımlık bir sıçrama ile yüzlerce adım süren bir çöküş ayırt edilemiyor. Son okuma da (0,4403) dalganın neresine denk geldiğine bağlı; son dört nokta 0,4311 · 0,4201 · 0,3533 · 0,4384. Doğruluk üretim ister, pahalıdır: her 100 adımda sabit 2.000 + 2.000 soruda ölçülür. Kayıp zaten her adımda hesaplanıyor; GPU'da birikir, adım başına senkron yok.

**Sıra:** `0 HAZIRLIK` → `1a` ve/veya `1b` (tek düğmeli: `1c` bitince `1d`, sonra `1e`) → `3 NABIZ` · `4 EĞRİ` → bitince `5 SONUÇ`
**Çekirdek düşerse:** `0 HAZIRLIK` → `2 SÜRDÜR`

**Dönen hücre YOK.** Koşu arka planda bir iplikte döner, her hücre hemen geri gelir (kural 8).

In [ ]:
# 0 HAZIRLIK  |  CPU  |  tekrar: GUVENLI
# Cekirdek dustuyse ONCE bu hucre, sonra "2 SURDUR".
import os, sys, subprocess

from google.colab import drive
drive.mount('/content/drive')
KOK = '/content/drive/MyDrive/model_17'
VERI = '/content/drive/MyDrive/model_15/veri_cok.pt'
os.makedirs(KOK, exist_ok=True)

# Kod her seferinde TAZE cekilir -- Colab'da elle duzenleme birikmesin.
DEPO = '/content/sekerai'
if os.path.isdir(DEPO):
    subprocess.run(['git', '-C', DEPO, 'fetch', '-q', 'origin'], check=True)
    subprocess.run(['git', '-C', DEPO, 'reset', '-q', '--hard', 'origin/main'],
                   check=True)
else:
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/sekerahmet/sekerai.git', DEPO],
                   check=True)
SRC = DEPO + '/deneme2/model_17'
KOD = subprocess.run(['git', '-C', DEPO, 'log', '--oneline', '-1'],
                     capture_output=True, text=True).stdout.strip()
print('kod   ' + KOD)
# Yerel commit GitHub'a gitmediyse burada durur, ESKI kodla kosmaz.
assert 'kayip_iz' in open(SRC + '/train_17.py', encoding='utf-8').read(), (
    'depodaki train_17 ESKI -- yerelde git push gerekli')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
for _m in ('model_17', 'train_17', 'olcme_17', 'veri_t17', 'veri_mat17'):
    sys.modules.pop(_m, None)          # taze kod gercekten yuklensin

import torch
import model_17, train_17
import veri_mat17 as VM

# Kural 9: veri Drive'dan; iz yuklerken YENIDEN hesaplanip karsilastirilir.
eg, tu = VM.yukle(VERI, 'cok')
EG = VM.pencereler(eg)
OLCUT = VM.olcut(eg, tu, aygit='cuda', en=2000)

# Kullanici, 23 Eylul: "lr 0.002, yığın 4096, çok u yapalım ... 16.000 adım".
# bas = yedek = 100: taban 400'de bir olculmustu, dalgasi okunamadi.
ORTAK = dict(lr=0.002, yigin=4096, adim=16000, bas=100, yedek=100, tohum=0)
KOSU = {'MAT_COK_DT16': dict(genislik=16, durum=16, bellek=64),      # es boy
        'MAT_COK_DT256': dict(genislik=256, durum=64, bellek=1024)}  # kucuk
EK = dict(veri='model_15 veri_cok.pt', iz=VM.IZ['cok'], sinav='MAT',
          onkayit='belge/onkayit/model_17_MAT.md', kod=KOD)


def bellek_gb(genislik, durum, bellek, blok=model_17.DT_BLOK):
    """Ileri gecisin geri yayilim icin SAKLADIGI, TAHMIN (olculmedi): her
    konumda S (durum x durum), bellek aktivasyonu, artik akis."""
    B, T = ORTAK['yigin'], EG[0].shape[1]
    return blok * B * T * (durum ** 2 + 2 * bellek + 6 * genislik) * 4 / 1e9


n = EG[0].shape[0]
print('veri  %s   iz %s   kapi GECTI' % (os.path.basename(VERI), VM.IZ['cok']))
print('egitim %s soru   tutulan %s soru   pencere T=%d'
      % (f'{len(eg):,}', f'{len(tu):,}', EG[0].shape[1]))
print('1 epok = %.0f adim   %s adim = %.0f epok   wd %s'
      % (n / ORTAK['yigin'], f"{ORTAK['adim']:,}",
         ORTAK['adim'] * ORTAK['yigin'] / n, model_17.WD))
print()
for ad, a in KOSU.items():
    par = sum(p.numel() for p in model_17.DT(VM.N, **a).parameters())
    print('%-14s genislik %3d durum %2d bellek %4d   parametre %9s   '
          'saklanan ~%.2f GB' % (ad, a['genislik'], a['durum'], a['bellek'],
                                  f'{par:,}', bellek_gb(**a)))
print()
print('ORNEK  (koseli = hedef, kayip YALNIZ orada)')
W, M, H = EG
for i in (0, 1, n - 2, n - 1):
    print('  ' + ' '.join(('[%s]' if h else '%s') % VM.AD[t] for t, h in
                          zip(W[i][M[i]].tolist(), H[i][M[i]].tolist())))

In [ ]:
# 1a KOSU BASLAT -- DT ES BOY  |  GPU  |  tekrar: degil -- ayni adla ikinci kez
#     calisirsa eskisini <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
# Eski mimariyle AYNI BOY (7.860 / 4.529 parametre); karar kurali bu kolda okunur.
AD_A = 'MAT_COK_DT16'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_bos = torch.cuda.mem_get_info()[0] / 1e9
_ger = max(2.0, 2.5 * bellek_gb(**KOSU[AD_A]))
assert _bos > _ger, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_bos, _ger)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB'
      % (torch.cuda.get_device_name(0), _bos, _ger))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme -> "3 NABIZ".
print(train_17.baslat(AD_A, EG, VM.N, olcut=OLCUT, aygit='cuda', kok=KOK,
                      ek=EK, mimari='dt', sozluk=VM.AD, **KOSU[AD_A], **ORTAK))

In [ ]:
# 1b KOSU BASLAT -- DT KUCUK  |  GPU  |  tekrar: degil -- ayni adla ikinci kez
#     calisirsa eskisini <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
# Varsayilan DT boyu (1.207.044 parametre): "boy yetiyor mu" sorusu, kiyasa GIRMEZ.
# 1a ile AYNI ANDA da baslatilabilir; ayni GPU'yu paylasirlar, ikisi de
# yavaslar (olculmedi).
AD_B = 'MAT_COK_DT256'

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_bos = torch.cuda.mem_get_info()[0] / 1e9
_ger = max(2.0, 2.5 * bellek_gb(**KOSU[AD_B]))
assert _bos > _ger, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_bos, _ger)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB'
      % (torch.cuda.get_device_name(0), _bos, _ger))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme -> "3 NABIZ".
print(train_17.baslat(AD_B, EG, VM.N, olcut=OLCUT, aygit='cuda', kok=KOK,
                      ek=EK, mimari='dt', sozluk=VM.AD, **KOSU[AD_B], **ORTAK))

In [ ]:
# 1c KOSU BASLAT -- A: BELLEK 512  |  GPU  |  tekrar: degil -- ayni adla ikinci kez
#     calisirsa eskisini <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
# 1a'dan TEK fark: bellek 64 -> 512.  Soru: darbogaz cozumleme kapasitesi mi?
# Onkayit: belge/onkayit/model_17_MAT.md, "TEK DUGMELI KOSULAR A ve B".
AD_C = 'MAT_COK_DT16_B512'
KOSU[AD_C] = dict(genislik=16, durum=16, bellek=512)

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_bos = torch.cuda.mem_get_info()[0] / 1e9
_ger = max(2.0, 2.5 * bellek_gb(**KOSU[AD_C]))
assert _bos > _ger, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_bos, _ger)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB'
      % (torch.cuda.get_device_name(0), _bos, _ger))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme -> "3 NABIZ".
print(train_17.baslat(AD_C, EG, VM.N, olcut=OLCUT, aygit='cuda', kok=KOK,
                      ek=EK, mimari='dt', sozluk=VM.AD, **KOSU[AD_C], **ORTAK))

In [ ]:
# 1d KOSU BASLAT -- B: YANSIMA 2  |  GPU  |  tekrar: degil -- ayni adla ikinci kez
#     calisirsa eskisini <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
# 1a'dan TEK fark: yansima 1 -> 2 (token basina iki yazma/silme).
# Soru: '+' yuvalari ayirmak icin kullaniliyor mu?  1c BITINCE -- ayni anda degil.
AD_D = 'MAT_COK_DT16_Y2'
KOSU[AD_D] = dict(genislik=16, durum=16, bellek=64, yansima=2)

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN; yansima durum izlerini katlar
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_bos = torch.cuda.mem_get_info()[0] / 1e9
_a = KOSU[AD_D]
_ger = max(2.0, 2.5 * _a['yansima']
           * bellek_gb(_a['genislik'], _a['durum'], _a['bellek']))
assert _bos > _ger, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_bos, _ger)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB'
      % (torch.cuda.get_device_name(0), _bos, _ger))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme -> "3 NABIZ".
print(train_17.baslat(AD_D, EG, VM.N, olcut=OLCUT, aygit='cuda', kok=KOK,
                      ek=EK, mimari='dt', sozluk=VM.AD, **KOSU[AD_D], **ORTAK))

In [ ]:
# 1e KOSU BASLAT -- GECICI BELLEK  |  GPU  |  tekrar: degil -- ayni adla ikinci kez
#     calisirsa eskisini <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ
# 1a'dan TEK fark: gecici bellek, W 16, 4 kafa (TASARIM.md §GECICI BELLEK).
# Soru: sira ayractan okununca birler sutunu kalkiyor mu?  Onkayit: "GECICI BELLEK KOSUSU".
AD_E = 'MAT_COK_DT16_GB16'
KOSU[AD_E] = dict(genislik=16, durum=16, bellek=64, gb_W=16, gb_kafa=4)

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_bos = torch.cuda.mem_get_info()[0] / 1e9
_a = KOSU[AD_E]
_ger = max(2.0, 2.5 * bellek_gb(_a['genislik'], _a['durum'], _a['bellek']))
assert _bos > _ger, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_bos, _ger)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB'
      % (torch.cuda.get_device_name(0), _bos, _ger))

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme -> "3 NABIZ".
print(train_17.baslat(AD_E, EG, VM.N, olcut=OLCUT, aygit='cuda', kok=KOK,
                      ek=EK, mimari='dt', sozluk=VM.AD, **KOSU[AD_E], **ORTAK))

In [ ]:
# 2 SURDUR  |  GPU  |  tekrar: GUVENLI
# Cekirdek dustuyse: once "0 HAZIRLIK", sonra BU hucre.  AD'yi sec.
# Kural 1: uzatma SURDURMEDIR -- uzatmak icin ADIM'i buyut, bu hucreyi calistir.
import os, re, torch
AD = 'MAT_COK_DT16'          # KOSU'daki adlardan biri
ADIM = ORTAK['adim']

# --- GPU KAPISI (CLAUDE.md kural 2), esik HESAPTAN
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_bos = torch.cuda.mem_get_info()[0] / 1e9
_a = KOSU[AD]
_ger = max(2.0, 2.5 * _a.get('yansima', 1)
           * bellek_gb(_a['genislik'], _a['durum'], _a['bellek']))
assert _bos > _ger, 'GPU da %.1f GB bos, gereken ~%.1f GB' % (_bos, _ger)
print('GPU kapisi GECTI: %s  bos %.1f GB   gereken ~%.1f GB'
      % (torch.cuda.get_device_name(0), _bos, _ger))

_d = KOK + '/' + AD
_n = sorted((int(re.findall('[0-9]+', f)[0]), f)
            for f in os.listdir(_d) if re.match('t[0-9]+[.]pt$', f))
assert _n, 'surdurme paketi YOK -- ' + _d
SON = _d + '/' + _n[-1][1]
print('son nokta  %s   adim %s   hedef %s' % (SON, f'{_n[-1][0]:,}', f'{ADIM:,}'))
assert _n[-1][0] < ADIM, 'zaten hedefe varmis -- uzatmak icin ADIM buyut'

print(train_17.baslat(AD, EG, VM.N, olcut=OLCUT, aygit='cuda', kok=KOK,
                      ek=EK, mimari='dt', sozluk=VM.AD, surdur=SON,
                      **KOSU[AD], **dict(ORTAK, adim=ADIM)))

In [ ]:
# 3 NABIZ  |  CPU  |  tekrar: GUVENLI
# DONMEZ, hemen doner.  HICBIR SEY KOSTURMAZ (kural 8) -- yalniz gunlugu basar.
train_17.nabiz(30)

In [ ]:
# 4 EGRI  |  CPU  |  tekrar: GUVENLI -- KAYITLI sonuca bakar, hicbir sey kosturmaz
# Ust: SAYI (her 100 adim, sabit 2.000 + 2.000 soru).  Alt: HER ADIMIN kaybi.
# Kosu SURERKEN de calisir.
import torch
import matplotlib.pyplot as plt


def satirlar(ad):
    """gunluk.txt -> {adim: (kayip, egitim, tutulan, ilk, uzunluk)}.
    Surdurmede tekrar eden adimda SON yazilan kalir (yorunge ayni)."""
    r, yol = {}, KOK + '/' + ad + '/gunluk.txt'
    if os.path.exists(yol):
        for s in open(yol, encoding='utf-8'):
            p = s.split()
            if len(p) >= 9 and p[1].isdigit():
                try:
                    r[int(p[1])] = tuple(float(x) for x in
                                         (p[2], p[3], p[4], p[6], p[7]))
                except ValueError:
                    pass
    return r


def kayip_iz(ad):
    """Canli kosudan (SONUC) ya da diskteki son paketten."""
    k = train_17.SONUC.get(ad, {}).get('kayip_iz')
    yol = KOK + '/model_' + ad + '.pt'
    if k is None and os.path.exists(yol):
        try:
            k = torch.load(yol, weights_only=False,
                           map_location='cpu').get('kayip_iz')
        except Exception as h:          # yazilirken okunduysa
            print('  %s paketi okunamadi (%s) -- tekrar dene' % (ad, h))
    return None if k is None else k[~k.isnan()]


fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
for ad in KOSU:
    r, k = satirlar(ad), kayip_iz(ad)
    if r:
        x = sorted(r)
        a1.plot(x, [r[i][2] for i in x], label=ad + ' tutulan')
        a1.plot(x, [r[i][1] for i in x], ':', label=ad + ' egitim')
        a1.plot(x, [r[i][4] for i in x], lw=0.6, label=ad + ' uzunluk')
        s = max(x, key=lambda i: r[i][2])
        print('%-14s %d nokta   son adim %s  tutulan %.4f   en iyi %.4f (adim %s)'
              % (ad, len(x), f'{x[-1]:,}', r[x[-1]][2], r[s][2], f'{s:,}'))
    if k is not None and len(k) > 200:
        a2.plot(k.numpy(), lw=0.4, label=ad)
        # Sicrama = kayip / onceki 100 adimin medyani.  Tek adim mi, cokus mu?
        med = k[:-1].unfold(0, 100, 1).median(1).values
        o = k[100:] / med
        en = o.topk(min(5, len(o)))
        print('   en buyuk sicramalar: ' + '   '.join(
            'adim %s x%.1f' % (f'{int(j) + 100:,}', float(v))
            for v, j in zip(en.values, en.indices)))
a1.axhline(0.4403, color='gray', ls='--', lw=0.8, label='taban tutulan 0,4403')
a1.axhline(0.3693, color='gray', ls=':', lw=0.8, label='taban 3 terim 0,3693')
a1.set_ylabel('SAYI (birebir dogru)')
a1.legend(fontsize=7, ncol=2)
a1.grid(alpha=0.3)
a2.set_yscale('log')
a2.set_ylabel('her adimin kaybi')
a2.set_xlabel('adim')
a2.legend(fontsize=7)
a2.grid(alpha=0.3)
plt.show()

In [ ]:
# 5 SONUC  |  GPU  |  tekrar: GUVENLI -- kosu BITTIKTEN sonra, sorularin TAMAMI
# model_15'in kayit.txt'siyle AYNI tablo: terim ve cevap hanesine gore,
# egitim ve tutulan.  Taban ayni veride, ayni olcutle.
AD = 'MAT_COK_DT16'          # ya da 'MAT_COK_DT256'

# --- GPU KAPISI (CLAUDE.md kural 2)
import random
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
print('GPU kapisi GECTI: ' + torch.cuda.get_device_name(0))
_t = train_17.IPLIK.get(AD)
assert _t is None or not _t.is_alive(), AD + ' HALA KOSUYOR -- egitilen model olculmez'

R = train_17.SONUC.get(AD, {})
if 'model' not in R:                  # cekirdek yeniden basladiysa: diskteki son paket
    k = torch.load(KOK + '/model_' + AD + '.pt', weights_only=False,
                   map_location='cuda')
    m = model_17.DT(k['n'], genislik=k['genislik'], durum=k['durum'],
                    blok=k['blok'], bellek=k['bellek'], yansima=k['yansima'],
                    pay=k['pay'], gb_W=k.get('gb_W', 0),
                    gb_kafa=k.get('gb_kafa', 4)).cuda()
    m.load_state_dict(k['agirlik'])
    R = dict(k, model=m)
m = R['model'].eval()
print('%s   adim %s%s' % (AD, f"{R['adim']:,}",
                          '' if R.get('biti') and not R.get('durduruldu')
                          else '   UYARI: BITMEMIS kosu'))
print()


def _h(x):
    return ('%7.4f %7.4f %7.4f %7d' % (x['sayi'], x['uzunluk'], x['ilk'], x['n'])
            if x else '%7s %7s %7s %7d' % ('-', '-', '-', 0))


TABAN = {2: 0.5121, 3: 0.3693}
for olcu, baslik in (('terim', 'KAC TERIMLI'), ('hane', 'CEVAP KAC HANELI')):
    e = VM.kirilim(m, eg, aygit='cuda', olcu=olcu)
    t = VM.kirilim(m, tu, aygit='cuda', olcu=olcu)
    print(baslik)
    print('%8s%-33s%s' % ('', 'EGITIM', 'TUTULAN'))
    print('%6s  %7s %7s %7s %7s  %7s %7s %7s %7s'
          % ('', 'SAYI', 'UZUNLUK', 'ILK', 'n', 'SAYI', 'UZUNLUK', 'ILK', 'n'))
    for a in sorted(set(e) | set(t)):
        tb = ('   taban %.4f' % TABAN[a]) if olcu == 'terim' and a in TABAN else ''
        print('%6s  %s  %s%s' % (a, _h(e.get(a)), _h(t.get(a)), tb))
    if olcu == 'terim':
        top = lambda d: (sum(v['sayi'] * v['n'] for v in d.values())
                         / sum(v['n'] for v in d.values()))
        print('%6s  %7.4f %23s  %7.4f %23s   taban 0.4403'
              % ('TOPLAM', top(e), '', top(t), ''))
        print('        BITTI satiri: egitim %.4f  tutulan %.4f'
              % (R['egitim'], R['dogrulama']))
    print()

print('GOZLE  (tutulandan 12 soru, tohum 7)')
VM.goster(m, random.Random(7).sample(tu, 12), aygit='cuda')

In [ ]:
# X DURDUR  |  CPU  |  tekrar: GUVENLI
# Bayrak koyar; iplik bir sonraki adimda CIKMADAN ONCE Drive'a kaydeder.
# Durdurmak veri kaybettirmez -- "2 SURDUR" kaldigi yerden devam eder.
train_17.durdur()            # hepsi;  tek kosu icin: train_17.durdur('MAT_COK_DT16')

In [ ]:
# Y DRIVE'DAKI KAYIT  |  CPU  |  tekrar: GUVENLI
# Kaydi KOSU KENDISI yaziyor; bu hucre yalniz NE VAR diye bakar.
for ad in KOSU:
    _d = KOK + '/' + ad
    if not os.path.isdir(_d):
        print('%-14s henuz kayit yok' % ad)
        continue
    _f = sorted(os.listdir(_d))
    _b = sum(os.path.getsize(_d + '/' + f) for f in _f)
    print('%-14s %3d yedek   %.3f GB   %s'
          % (ad, sum(f.endswith('.pt') for f in _f), _b / 1e9, _d))

In [ ]:
# Z GPU DURUMU  |  CPU  |  tekrar: GUVENLI
# Kosu SIRASINDA GPU-Util dusukse darbogaz FLOP degil cekirdek baslatma:
# durum_gecisi T kez SIRALI doner.
!nvidia-smi